# Brazil FDI Observatory — Python robustness laboratory
**Prepared extension; not a completed empirical sensitivity study. Fernando Barbosa.**
Use a standard free CPU. Do not connect a GPU or purchase compute. This is a new notebook;
it does not run the Financial Intelligence models, LangSmith experiments or the baseline 2,000-draw bootstrap.
The baseline study and all prior archives remain read-only.
Read Research & Methods before interpreting responses as effects.

## Questions and assumptions, before computation
1. Are remaining residual patterns seasonal or serially dependent? Test squared-residual quarter indicators (seasonal volatility) and auxiliary residual-vector lag dependence; report the entire equation family with Holm adjustment. Quarter-mean tests would be tautological after fitting quarter indicators.
2. Do LP responses depend on additional lags? Compare 0, 1 and 2 additional lags with HAC bands. These reuse generated baseline shocks, omit some uncertainty and do not implement a new identification design.
3. How do units and growth accumulation change interpretation? Normalize each saved draw to a +1 percentage-point FDI/GDP impact; accumulate GDP growth and convert log differences to level percentages draw by draw.
4. How does a finite-order model behave when the true system contains moving-average dynamics? Repeated synthetic DGPs, 119/240 observations, VARMA coefficients 0/0.5, fitted lags 1/2, 1,000 deterministic replications per cell. Report bias, coverage, Monte Carlo standard error and failures.
These are prespecified for this new extension, after the original baseline was observed. They are not retroactive preregistration.
No municipal treatment registry currently supports an empirical ATT. HonestDiD-style sensitivity, a municipal Monte Carlo battery, alternative inflation deflators and new omitted-variable specifications remain separate unfinished work—not claimed implemented here.

## Books and papers actually informing the checks
Pearl & Mackenzie (2018), *The Book of Why*, Chapters 1 and 4: association versus intervention; common causes, mediators and colliders.
Freedman (2009), *Statistical Models: Theory and Practice*, Chapters 1, 6 and 8: stability under interventions, economic units and bootstrap assumptions.
Verbeek (2017), *A Guide to Modern Econometrics*, Sections 2.5.7, 5.2.4 and 9.2: power/p-values, reverse causality and Granger prediction.
Matheus Facure's Python handbook: graphical causal models, doubly robust estimation and staggered DiD.
LP interpretation/inference: Plagborg-Møller & Wolf (2021), Montiel Olea & Plagborg-Møller (2021), Li et al. (2024).
Municipal design: Callaway & Sant'Anna (2021), Sant'Anna & Zhao (2020), Sun & Abraham (2021), Roth (2022), Rambachan & Roth (2023).
The workbench bibliography includes DOI links and the two multinational-investment applications. Private books are not embedded or redistributed.

In [ ]:
from pathlib import Path
import base64,hashlib,io,json,sys,subprocess,zipfile,shutil
PROJECT=Path('/content/brazil-fdi-robustness-source')
raw=base64.b64decode('UEsDBBQAAAAIAAAAISg35XcoQBQAAKkzAAAZAAAAc2NyaXB0cy9mZGlfcm9idXN0bmVzcy5wea1bbXPbtpb+nl/Bm34AGUO05cTZlip3mjTpvZ3x3E2TbHdmNRoOTEISY76FIG3JXv/3fQ4Avslyk925mkxNgsDBwcF5ec4B+vz58zdJkjZpWYjM+fXDf87KIts7dXnVqqaQSjmqERupfOetUDJLC+ms00wqR9TSqaVIdH//2T9LR8Rf21RpUtxJi7WsZRFLJy8TmSnuXIpi8ylPm60TiyxTTlk7Vx3Jq7JsVFOLypE7GbdEwn/2/PnzZ+u6zJ1KNNssvXLSvCrrxvmA12f2WdSbStRKcmcrFHXizhdF86stqGTcuUsr4pfbwejh57IRiWhER6No82rvCOUUVddUiSJBA/5VSdcGOTTKLMYXVUofVW4YHH/Sz37eZk3aSNV0TOuGKpPUpsyodZL6ZlDXKZEq3YD5ddpEaq8amXPIWFVloSQkmJWQXFTV5RcZk4jQVGUiLZ49+/PNx9/fvL18/ylcsmIdp4yzTVJFm7q8bbb0so7Xw1stZR3FW+yHxBvYiNCZrZ59ev/+XXh+dv56fvZy/iyRa0hRuJUXgImmrYtOxj6azy9eu7QR+OyTGkRXeyzN9Tx/K3dJCo1pXM/QEDfSrfiNyFrpBc8c/KrQDl1UPrZPFo2fXydp7ZoXFX6uW8nlLlVNVF7rN08PrMVtSPvrJ9gz5erVu4YyV5BgdC33dnRaJCAVnnMoW3kbFaIIfxOZkp4PrYTUXUMRMq7Cyr+FXkaqXa/Tncv8Jq+Yt6BP/m2NbbSLw+y2tZaYOsayDBGhlMT2ac6yUiTgbCqUMByzqsWSFlXbgGZbWKHgyYiFmnTLOqwSQydWN9R8yirMDBklp19bUTeyhunhG9Or3UVxmYVnZmwejrgxY42unRbCGLtPHZjdvUbuGlcLJi02IWub9Y/MmywuXzJSbclWYcjiMte6zBwYCn3qDZitlgyrTBPqd352djamkWFD3bWvmYVM9CuWWMk6LZOoJoXsPi/PVrx7nM3xXMuvIfsD/ArV7CvpYrYpg5ks3DWozuc/aa4GQmB4/tNPr/84Z5MPIIsv0PeLP16xMaWi8kVdi30kIeUMHFlh+00ZaWfhehxsf9Bs/060Oq5HbNoxhsV9uF72Nroa6KyxP43pchdiWmxgmxcRBB1fu8v1cslaFU0seS2TdVskqrPf1SNqxNpGNhFMJE+hfT37PKnLKlqntWqMSR2OXHkHUkjVOi2g/+4eYs8y19PyG3+4sx/0wB+cjzKGW2rqNm7IcaUJ5Kcc7e0+vfnz/TsnLiVsLE7JyrlTlPDgTiFvh0gA1+drYrvQeEN3z+84NCwTG7byFm24X3ZvwWq2+4U2S+ntctE+ps/setS2jK8ViRcziGzjqzKDS5oOhAcWcRPloqnTHUby1v/s+Z+NZRrvt+Y5J2ZabihqM4ZdF2ReHNw2nPxCGmsDU+Ec2n/MthfU1bzSk2EyXWtpIH5CfgJx0x3Tgn03HkXMcePP86AWqZLOn+RV3td1WbvsQ0kR+EaOe0LqbUEbgvgM38Em20wswAcYmXh/C8Hh8Gr2uzzshf2PapkJmihqSncyxvouCOtFFI68nG6mCCJVeK8CCi3klpTnrLEwxMDCWT7p3447r9WDJgqdW6eb8J7dyFrhMwvmnCmJlQYU0jjrtIsFZn7OKAJEJowxzUoUEUyIIo+zRFaSgke8j6gbC+4rvxC5DEw01OxWml29h/1IG8pOKaKeWjRQ7Rn/i16qKWugK+q2euAMKnhNYIumDB4DFt+ub8oE0ybMaDSBFjyoONUNI1iiyW/LOr2D4rBAe17jb+cvPY9rMXY/llURwkEtIlgZOi/P+JyfrzBexNEVJrlNE/iigG2dk8FwK7zoUY4edUCycwZwWxKmg8H/ePOro2iXZTLrvjp212dwpqS7WCMpoGh3aZaKet/7lNkNEBC+Yirnv0SWOEkqNkWpmjRWC+cfZZY7FNEhHiniLZzKjZyRM9fWsBZ5mu0POVQpIJruAOmzsZ1FFcElmWUsmNgkQ88log0/f4VgxXJBovK5f4EXQnBWeiS7h8PJsDMtSaHT94X2ggZqd4YKcdJsaTOAYmxqDCCR09BPstCmnjZ7x0jVOFQilBL2SdeWVd8BOp942T5c+8yYkBnQ7MMDiDcCW8bKDmDWgKamwE87PUxCTrw57agvg/nr1YLavxPvaexI/YF84GiN1XPoRtOx0zPePfDyCn7tRgJPlLcqNKjAOiVLihDvkOCQ8XkTnHdgsBMkNw4IRE0HgQPtHoG6w5gxdoiLKuwi2eJrOKCMFDh/WQWjwG5mtd/Dg/j+9Ztx3QbnP8x4J5eiMPmbyGhtewq6jUwI5s8oPwEEdJptichiut6WLUzsSjqNaJsyKzfQq6yL0o9QCxpKyNUlyX+Fb7FsI3YrKbB/4XKFpzoVGZ40EbilDUXmccRHbpCE85M7UsRKLuerxXW471+MbOEAvnDyzOQHJVYsawBUtwdadg9037QJVe7/x+Unt10G/MvqxYtzvvN8tEOTbiKClCH5JMbp9fo2QZCCSe+MEb/iAGIS4aCuTerFApL0g9dPcBsScrmFL4pIhIQu5F66r7zlPFhxBYGJ2qg5EVrbXKQfbkSDvJJCj4tIJiCgqwzBh9ZnXHlK7o0Fek/dW79vopC1ZsFLRIDha2VSjRGHmDcc+f3E807cZXKCBb641rIkZwpBms+VtyJM8iUMzxzED+mMh/LkpHpx3VlVtxeXOhy94j+ugom3A92NTB4rSru8nOFbQIrSevS4eszHnF+ezDtk2v1quXlMrtOi5WVAMl5xM+/BUMJZPRA0aI8Sj2vArQ2hH7nplewIuvqInjNYvEGYvd2Pwg+bTofQNejdJSkep4m+V+8uv6l3Rhzw5an+TmK5k3WJnO/6xSUfL8fzFqOOMIIZegSr0GoqXqZUb0MwP9Lo0eBv67PRaTLyp3WaPB+t8Dt0m9ZyRLs5Ix/POg26tHIhJcIHXt2QHt2llWvMi0+KMO6yBt5kRudq6ml6rTjw1rZMQuClLGceiS4AuaV+jzAiNGxUN8N0x3Tf4IxwWY9nIJGQEtbW84fh5WpiRVPGDY1vMW56/Z8Zt2Hs3ib2wZDWE3wmWSCkFS0BH6ZT4R6fqUrGBC90gOhg3Cjjc7vhzk1JoCojkKLiWsrC69I+IB8pGkcXycq1jUHdDCbyPMJoWnwdS0g8rW95hAvHaaBD9uB0UHDAkh0epBJbI5DOIuqljeqjEXkZ0AXeOcKHWR1SB6NYljN6p79AacLgNYNzhdrnVVNCn0fzH+JWjVidjSx0FBtJc+GonNaqBO2OySygSZnxCE6MHUhSyiQaR6V3ktAe5Iv17pDRmrEzM9YxRUUrc4hJtIr2iOo1BikCDhKgQQagBnj5/8AzZdt00Z201CQGZCImlxiMBN3D6kR/X0AN6yZES/91UgTAB667eAtZqdAwsNQtM8ClYdARULLjGAIIQpk38AfSaWDEAYF0fG5HgeelNw1jhcaSIOPNto9Cys/hbtd72pOLI6Hj90KXFw8DB+kYID8ZAB6SMj8IILBfYjhOx6x27H4Z2L044FZ3GUDP3sjpZBvYvycFxaLdbhkUq+8OR1tCC98VkizrnfMnbASsL3K1nF0E2Ic4HX8iLB9hT1yRVVsR+mcXnuk3oUklCDsIpL1+E7vGOB36Q/+GwDNOZvUzZ2mNqEIVG1aVoHOL3Yp+uoiQ4bKgo/vwHV7S+kGTEbIA0x64iquyRVZe7ylhI6Tap2CDoRtFXlAZoz+IIa9R5uSO9NcZofLcpK9tEcuaHFaz951LYCXRbnIolfkqMoBvJymhU+RmqWu63gPOSweZJGZrixSqk8+GAxp8K2uZG79BZUxT/mgOs0jrGygnS6KkFreRQgqCoC4PM55p4WcRt8jHlFbj1iTZN3LsHdJBjc+Hmhn9tEskvFaKxO0LaKdrphMtvXVDTWjWZ7Wn92lwdvHgF9Uds/X/Ko2vM2lhCp3dxFN7gUKE8VLrBdyDLgaGeIG/esVfrR7Zezmtj5oBukJnHn8O53I2PzvqB7S7BYkanjm9MyWHqaYbgXUKTGwEPAAbp908P/THQk5TIpaezJ2qcn579/vp3999sCwcUOwE31El2Ler8jk94Cu20k40x0Te6Rz78IL+Yw5sSNLhQcWURPUdJtKriwVqZnHAboW87drPBqthUNAm0loPQv3pl1MBFNAyZR2Tsm+kYWpYdFGUN1qYi1620sR8msS52pvofltY+bDDOYEHoO6ZLr9BjZC8Blhx12r55l+52KUqPDMluK86qPln5xfcn7/m/o+vuP/Tv12MCz76JC6TNzKLLPMsOCJ8vZpH4j9OZuSynuK12+6/4PeQz5Gr+rUfPyiaDhzKEY2Vn+NuQdV33kG6M+IGYi/ItQnCX4WtVS4w4RagCCN71zf1K6fI10cFKe2FdDGuEYUsWxpXJKr3Pn2ZLhoV49yCw4s1gldUgrcO5Afn8xYxddblHQ7SFzgZDK8dc0bR1nC17/7+YYFPrTSu1kmVgyCfZoQMyc4tMCUuu7Btah91obNQOC6EbR/MCfAcodUdWj8hpn8C0iRH6y51NbqgYr6r2dUGZnhe2WT6TdidPrnLpf/6gp/xsxVf+vMz7r/Cpp1d0NsZqds5919erKwJvh2Pm/vdMPT50RLACIx63Q2QITj1jaW4hBtDtzi5OJvzl0gU930eGWXptXTlkOk049yc7HnvecF+2azCN7/gzwzw5+0vEu8nZon6ZWZLNvtwv8QcwWpxN8pUC36GOTWuDYejb1sG6k5H9Oclq0WStoqt/j2cB73bWYs0a2tyPC2dntBeD7vWVTjrNY/C3p+4mh6fn3uLrOJUGjk8XDcHT7YXH2qSUCtSlt4uAN16DZuTOx7pltGoczKabfjKqA0eFujRbEP3USWiKm9l7b7hW++Xt1Z+T/eZzdHL0/XlZxYfhVm13GqEvVBJqKR9mTppPTkL9B/OLLKQFg4pPKiEDkhgwHSCAsvMXHyazf2fXr9Qyc+hHvhziLYT23Z4gAB5RANdk3RScDHcwPWPXM2HzplZrz0j6HNJ/tygsAksAsDp6A64qUdSM91yOqRGI6hkTBmJ497BHldiQwM6p5KXmCyKRZ2VukT85DneD857CijjszW4ixht4GcEnTJAs9+MUuoLNHsyGsJg+AhYV+ijDVBBzmZ9iVRwHmqMigrtqrvzhVFZgYyQVMPkU/rAYYplhrOhaa7V/XRh/CCd6MaBqcHAx2I4kl/QD0FhI5vQFNbzmAGbFffFwywX95rJh1l1Xz1Q8z2oaVSmi/hHicHMDT1fnwUo94lJ6TcgwPFNBzt8UrJfUA41dF8yI2z2WADdzx6NjsfYs8JV+BcnJOaOB+Um33dKwtnH8SGtPjtocydPkbM38fa4jKgY+7RUmnpPJZ/wu8Lkk1TkLpYVAvyAW/nghi7T4k220a0aSMudnnHkhQmQyJ338CR9faxjtorfd9tBVHgn5+BfJ+UjmSn9yAjG6eSjThqmT6p3NIR0tF+pObLHB2Q25C+BSimsUe1KF+k6Jzoq1WmqK4/ImDxAF9f/iaTtsZkanzDksUi1C3POaHaRKqIVXecCvkDETA7PJ809HA269SMUznJuoTgtyJuNvh+VFBt54KhbJwu6J3AUq97NY/nqa62rCfrrC3c+656902Eq75gInpj/KhVqRL8Xbx9iVjO8mbD2WNTHJjqiE6OjRh0OZjoczBAsdBXQHj3aPfmOtGdf0CZRUVtXyRkdICvaIU1gEgI/6fKeqXRJOmgr4i3y6muDP8EBIAb+2ZKL04lTr/RtLe7SzDdQtwci+ijYyZHWxGlFZSa6PeB81AHHWfeBic6ewTlFq89bBLJ+5qGAkNKKqLygwxcF0BFZGp5lMps1NTSUCA5H0qppk30XYK9EfN1WWrzdjRg6HDYZPTWaO2syv0LWEN7T7bnR7RLqgMwTyEelO8SE6eW6Idzpg+V6k5VXLnvB9L5Tjq4Pcu1dFminvuT3t5D5d2nFbCMdSKDJsDmDeIeqh95348pyUaRrQln3xaF/upm4HBO/ua7i20X5KSAtuDWURB1vqQhi9G04i54ZDjRruqMugNg7rP5/p9VvtBQ7mrNbxvtvv3+I3r3/7fLN5/fvtFu+myKG49wEd+bgmzw2egxWMWpnRH/Wrd3awcgfd18OvbH3zQUc8mnD7p1PE9Pph0foauIbbRclG/dO75o+lqQLltTUseL9z/0xrh8OJJJoiXRdOpHYKQ42+E7rnFtMY0sYJof3FJ7UIUS68YWjTgbwBKTHLLAN+uzL9fAnoqwM3tVsmfHY9gWj4jqm+yPEbaLv9IqEprbnIqafvW7l2F7TgqadzwBgCqnWMkUVdreq/Tf1piXj/0BvtestkKCLJImEbXfZbFa3WJsuHJM58+6WmVGCYwPKtqna5ukxlotH4/TtGMbjbZnGVE5kaUH110av3p6V4DmruqrTzFSX+Phuz4rbRD3shz854TiaWnaBBXsCOi9YiFDoS8xK0lg1vvui7wYKn24HCt+sGg8TUL24ktBF2V2I053/FVfiqpqKDOyT9fDj+/zdRZkA0YwcZn/BYCPRJHz9xNdZq7aj/YAvtZ/+NkhuMN1xGuDajifMXjHue30L35trUdOby1OKM9PFEj52N+fQU+j+Y/xOcjZcwIp+Nambpv4tBP74lvexbOPoNekJHbs3f2oPgcm7XkmXSHZ7YcnTBg1re4z/DaQI70dWEGQiv0pEcHgxSeuXp22k63Jw1Nf1GBtQ1/Vo5b8fMLo/Z/s/SqsPlP9hafd15R7cHBjlB2Z1R77/pWYcONt+vx8htYfJZVhrjU+bo+nwwFn/f8PkLZFM2NjsPtL/F0Pe2LnpNtmEBWzqGApNrAzGEUW011EE1Yki8slRxALjm5/9L1BLAwQUAAAACAAAACEoGBa1c48RAABXLQAAFQAAAHNjcmlwdHMvZmRpL21vZGVscy5weZ1aW2/cOLJ+z6/QZrGQZLPbdjDZnVFHQZzs5GSAxU5uZ1+EhiBLVDfTukWUOu4Y/u/nK5KSqHY7s3OCwC2RxapiVbFu1NOnTz9y2bUi7Xjm4KFPu75NCuc/1x+dpMockfGqE7ng2UJu63TnFHWK6aatv/C0E3Ull0+efN5yp+Vp30qx544omyTtnDIB2tsJgXQUAunUVXFw+irjrdNhYcbTImlBvTWMaKTvEylFtXEykWyqWnYilU5WA0tVdw4gk5tCyK3CAK67rVPneKkld3haV3Up0iOET58+fZK3denEcd5jkzyOidW67bBRIE0U3BMz9EXWFXO+JW0FJqRe2CTdthA3w6r3eB3Aq75sDk4C7pphqIH4MID/TabXy1Q0hyVY7+SAI92KZ2aShss644VcdjJRYF1dFyNokuV9UfCWObtGjnzay5JGEDlZaoxAUbfJhg8IWp5ksd6XTPbcPGZiAzExJ02quhLQ7ZMnGc+dpkhE5d36wRMH/0TuCGgDxKqUe7esapbYXNsmBz9oOYRZDQuWXQ29dJ7vn1zo0STr+qbg/rg00mv3vpPXrbN3ROXcrk8uz6DNcdkdtOvt/GC2esf0+qXoeCk9//4x/je84jCOB/zTuse4z4s66Xx1LsgKgUXIXFRYQpIyiP5dV1ytNu+3WqAZl2JTeQf2nTUMGNsuJEgjYD3QEEX1CLoKkQO9cj1k4wTltC76sooxle68CAOAll7BQcJfqAU++x6ph2DNzqKDfl4AIi6STWAgzetaCc+8kADbpNpw74o151f+eu3rPeSii+VBkoQe2UfFdiFMfJs0fPXf7uk2fCgbf4W/G96Fh2ELqyy8Ov+uUUdX2jxueJeE2Pp33tbSg/KGWbbz/VVa91Unw0jD0v6+TDvbGYbpXy95qIxWT2W+75w7XpSdQxZnOy0ZWyqNv6ZNfQnDS70XezHLzpszkPdH9DcsZpjcsZh4LUSVFJtlITv51buNAgbqa6Z3i7cva9bCf2VaqCMOkCMUfwlJbVgBa0sEKP8nKXr+a9vWred+BMACahKpgMt1lFMwZuda3EBmETCAUnhjZATP0fAq8wbkvrE1KbJQc7a4fUULAb8P1fjy8yv1e0Fr1JNeRNRYUW8y6G7ardQjHpaPJ4sgX4SXJzaCrS+aWoqOoonC3SPkYG3SCjqJZjfXRMDYv9oVFHa2C7Jzr4DRnu1YsF5+1tqb6U6vTmuEqQou3zIgKI4p5a3G2SggPASz1aSuxw00L6+CCW4XsGCxU5D8wIFrQTwYSSaZ6GWo3IcHgDK5pZ/kRnqTjLjY7JNCeiNK3x8Vof2dS7t0A/rL3Gs3uGYuhOKCiT1zBzlJN1CPzFUHxw3UD3MbN2iYm7kBZjQ/AFS/zL0RqRto7rSizmVfeto2/DNisd54k6J9S+k+MCdtUvKOt8BorQOZ+huGJth77UXw0iDSwlspE2XbuhXfoYerZ6xukRXY/mSnDF7BRWqza3+lgawzB4WRPtT43MmoIYWo4W056GbnR2pivXod0jhseVJDuq0LLncHj2Ze2ZRfaVhf/WqkW2FZj9nH+RXbKf+D2ehyPREd3dDWdrDjKsshGY/DrgmQI7fgbYIYY5i5dtfMhjbmuH0Zkl8nqtv1eXj9Sj0tyLnbVoTRV6/Za60KlczFVjKnfPADtWyTNHwL05pHK8OONrP1Kmavw2PVXhqnoBI/2x/UxZ57r5lBMdkuTqy//PwwLBjABlvX8UGD3IYPIuEt09RgKGMoQkRYIcn5ga4QAYbJuBA77gH8pMJOqatSRqrJ+ovtUeg63wbm97xar1q+CW/h9qv1uDyteW6ctnwQIQBvgoMdF2BX43JwCoWHNpZosSPfZ0cPqHBuMT+IhyOI6FAVhLJc/v6vT94sRPGNv8Q0OfS4OzQ8dN9dv3EZve6+ZTK8c+HjYHo4/9vzSXn3/gMikkdbikWa2PIG78S9bbKUn0quLXYrKKulPHWyC8N6E06EVidNdGYQ1+F0nlYqkzDvystSvmHesx8bN2/+rGU3gqdcWtaoczFjijeJ5EB2bJCHaSbSBrWGhZmn0VK7SaPa72vUlnJHJJ1yTW0cXSFP7NZrFeCjIFsfu6HjdHBuKBY+OJ0I8Isr4BpHJwc0/NPbn8MPY3NoC/L1GeSMeb3fiE4BM1aSiTznLUdicDLCjhnkYmRqGDD4zeuSgldyK2T4bAy9ODkT+pdXfPGPE/nKu9EmqZhF+Fa5S105eSIKFLZdTdVxXaV4c49COoZRo970yvsiRM84QmQ2LCPqHzHP6ITFA16hqteYE0MI8SPLJuR+26JGQX0wmG4Z/mzU2IcPLXXVh/2iX5Y8qbQ8Lv0VJfa9SexFtbcsHm9ejxDaX1T+6kN4uTwd58qZ5aRhH23JR73qo2CxXV9Uqw/nhLRrE5RaKSaA91VKf/0Lr1pstdw+nIXVWaXVnoe7s92ZVy6mY++fezvKu6aRubRhXW5QqtSoE9RTGNKeD0hXshySy5HP7Em3wwxV6EuZex9YllO2g0xnWyONcpPsSy+pa1L2BTJVyk077lClXSZVx5N+ZTcgFiN4Ig9l09UgDmvZtJxLal3keMjq0jUKs7oeXk7plQnKiswNEpQpEhN2jHYHu9KpsISkP8JPokceSggjAlmjXI9V78KbPPM30W3HzscyTbp0Gw+v3tHpH8GkKFHT56JAHui5CNt1C1tfJeHQtvAk0yEh/IklfVfTk/v6tzcuxWnqaACgJXlISUm5m7qsogASugRtFTD2lofC5c4dNuoGtC8G7eTxXJEJnD70R6SOZnZ65gbZ0I5DtTd1XSjwl8urS1XtE8SL5eVz3wQwdZ7Co1N1dLJtPlUWPr4xd1zpBuoRFtkgA2uRjA25+XAs9eta2aza39HEiyvKwKUkxh9MKeYVhWgw6/XLEBsZexhJdUAVOu5d9wFuyXRsjkk6yOqzpMVW3F+HBtvQ20sVJCwbHrRRngxiKBMgQSWge3V8NevkJS13KnJ1EiiddMuROjHFEJJRHAf8T5NeJgUpeTgUlWaoiHW/CxGuzw4wmqYwHMhZS6XuW4QEBbRs++rCbQBKncaLrz1cKG+LwzKVexf+rOmpUaBXjJ0nvTJNwF3muUfEXaZXjW2fsbvmWRQ17MWwdknzxpTVIQybbKkWgg9PkwfejN/GSGpDk2CQQUuzD0g+F5toNHfy1gfrQEvrROtulULx/WGerNdEbi/jTdbEGxRr3dZlbs5xYKtMxumWPLe7foCRgWlkgnHWl6Xgxj9FrhEqTDVr6ybORYuM+3Pbc/8YwXqMrTo7efHzqS7AW/5NdYiTyvn5UpfrLW3ZMYTkajQIyhBFqY1wy4vMiNjkgEgF5sJTeenQBsBBEBk8N/R/ur2l+4qNPhIP0WhzQWzm1EIPS1F5E0624wdUZeVNljhlUEaq0kZl0oTDgilI/dX5CP/ZOXSQisJJ9kge1H5r5MTtXhu44iWlNjcquZUD/2kkI2BZ1MjK6L3Es0zIIetgrMwwPN6ev3p3ZFWmuCGjEm1+IuV9p4VWNKok/FH1+I4p1a/o0IePRTNlsRol5fXhyeRey5eUrnPmtC8pUQODZ2fP2JCf5HyPSkVDRVfB+sJ6thI7KIQ3mSiltkzjsb/2nEJzaLsS1XKbveurB7uDacsOQaNDtE+a2F5ldKuciO2KlMdraoFibfQOixGFu1ILluUuE63HwXkX1zuL5Rwxl7dNi/WhbuB7Y/8ewVC5Joo5RM+C9YzPGrpBmvcBUL/RaMansYzH20Ru701CXFKhpLi7cLX7h4NWbm040wSxVDwjW1BxZnKNNOdHrsWSu/5LaL2e8AOvB6loKToDVacUsqTkZDjtw52GosLuZlQC64V6ZUbj1CszjwiwnAbmWlVjQ9mKACoyZFkryut7MseICnrqkj5MohcnKkA7ox5TNTFlySMzdqbcGHHn7p0ILp/fL6vmuzuwcDynKw5LIUYpaTOq5CiBo2RPdfiSzEsbny6Q0kDtdEiuvDRycdrg2PFQUJY9r+KphBFVz216hr2J6CCyAelkE2ZGNV2PELXVRtWoMKK6XCIFSJBqxxj1ptFP0NAnkhtd0USntMeEzbEqkaRuYkdAhfql4zAN6V2yqVVpPfpTPYokty8S8hREY5Y0HxXfzcPCe4bgYfU9az5EwdR7mBfPj9XkNuZzq7MxFNjW/MOKfL5YSwhwzQTVtYej9pHuDVkxZUSiYstxd1LDT7npy/DqxFnvK50sOqMrdDR2d47wJqb4FFvxSeNHgFrdxAhP8YngZDM4gh/zeXyvpyj5S0RkODMqrB7MF42ZPrGfigyR4KYNHe0Ee8MBb5Z0DmPZ57m49dxlVzbqmKPubpbk2L7HFOapOEIuSmsY2Aq1GIomVFysaHxJ0UeV0BiYH2MNrUCt43Cb8qZzvIlrNpX2/xLVdbFRo8ox8CMbCO9cK9i5gSDXmpDvCehulvv3q8krm3POcn917Azy2UWXJ2DQf3ssut4ol78Ow8ujozUSMtEJqlfVpHaG7O4Rr09yLbgaBV3mKpnpWwv1iGCpnaoeG1in4Ufiy1GTU0fe3H39+++fP33+eP3euQOh+4u7kYn7wLmb6N3rOMP02EjvHmH0i0oZXZYXvdxa6QCEptZMEsmEHNtK1jXZPrpcT5fsag2y0aKJH4W/egg/6Qqes6U7q1BdxX/1A6xE2gUvXnDPRsm+DomawvaVsEXLy2fP2fLq72z5809s+cs/nq/vR9Rg6cfYj3j+UwQoewss9PerObl7k9JX6gryqLNibpmOi4GEGh+UycEKFQh83KSOwU/NXNaQKOvCbanqPu/WnypwfUlFlxMjJ1PDgxahFv+dYALHPXeZ4y6/IKP01DLqdAAjIOZFOio8iuXBwMb9lIiYsHKqyLGupo7LCAD4q5P7y8kb/4B3JC1YfO+M1bijiI2MU7Nuanvk85bHqU381XlDGcSmr3uJY8cX9AkOp0aFLoZQMPI9lAex48Br8xVkLaagVMkqiRBevkqyL3CjVeeoQk7XUcCpa22trBfus8tnlx+uxtQX86rW8F/+/fIxkUUAWiPmq5/m/yk69721O4Yiua37zdZ5dnn1y4efJgnqbf85IQ7+MJzKovC/q3SU+CY/9jJc/vL8bMSikNd9hwIEYUPCRZdJDG1IFTmudE+WtjYw4Kq81bzoiks7Z93qRhmFjekmERgpEHd4zAuxEWqnqoc3ribOqAiNhlYZ9fumHorqGUqG9EN0eIvcf79985ujdOzoIg2k/uef752i3ji6U8KcvzlfL77S+Ns3b09OfPz1149qQndSzMTKQa3OQyi05akYzuRv1bekzZy3//zNubAOBGhimer+aVUGd8O1vmWHcOrM5aRxexBZH3Pt3oEJYASC3U8tk1jf0w/1Cr2s75nulTfqN9bdCqWp6I6Kx1LfxeovB4a+xnDjX45YlF8pyatMPZE105cedaFE/b+fqD4snLl4IZI3SmrqBjzPifqeOzmHawOw6lA5dCkOwA9aXHDYlHok9LFHRlZZt6pvqj9HjPXniG7wmshPV6XjvnXiPX1HEVvfmwww6jMAc2wol6KnKXC4U0yB1I7TT1KdWojs1N0mKfXhqKGa6UsbuihNUmYdLguvHZ5mndhPqs0/fqJpLtp1I5WsgJwG+TB9TBy+J1hs6J76fHsyGPwFO2PfBVyendErcxtyjpmcG9X8QjVYj5/7DaZGedLhARjoWT0guqBKNtZeSTZ/mJ1ZoydytDE5GitNZWaZMF1bmPnQcouVZVP2dE9F/+hkkbROL+w4bh41Bo5m1/MGuT5b8y9qR+/uWHxRt+/xj2Cdt/oC0e6cw23sSaWJUeoCK1AVU8dd+UDqlluuVn/WqF98Kxv/oyY1M0uG0DYoYmoqCbk90RWP/gjx2jTOkZBb24qHK4y5lzaqngcaS+8mZqmEThPuq7FxeoI7921f0JdkpMOexD3VmXRkJmt7tFx4YHr3s2sfLbUn/wdQSwMEFAAAAAgAAAAhKBPfHmL8BwAAjRcAABYAAABzY3JpcHRzL2ZkaS9zdG9yYWdlLnB5zVhdb+O2En3Pr1D9QrGrKO3i3kWhQA+LbRZdoC2K3cU+3DQQaImKWcsSS1JObNf//c6Q1JettEnRhyZAIonDmeGZmcMhF4vFu6Y2vDaXW65EKXgRaMPueZCzfMUDVhdB0Sq2rHgUiLrgksOf2lS7wMnjSMBUvhJbruPFYnFRqmYTZFnZmlbxLAvERjbKgKa6McyIptYX/tOK6VUlllHwm27qKGh0FMiKmbJRmyjQq9aICv7v4PNeyFJU3OkumOFGbHinuXuPAvy7b2ovJ5lB9Z3YL/B6cVHwMqibh5AmgeLgYd1Pj/FzpyFuTU5joRt0hpmQ2omFuOfahDCDJcFyZ7ge1PjFxHrFXv/3jZWh8Yo/+jlOQc7qphY5q8Itq1o+zEYE4qLdSO1GIg0uZ2u+0+lnBa+81ogm07kQ6XtWaR6xqmoesprV/l1zyRQzjdJpSCL4TQilMa/zpuAhaU15+R1xXmi25RlaDBGixCITBd6jiwB+8HsM6iDS8WZdCBW6l86bR6FN1qztG7UzzEamdtaDMKtMt2UpHq362D2/IjGIEHoNf+MHJQzPLIDhKST0utGx4pAIOQ9BOEIlzm/FWXHq9wmEVcMK7exaaWeDuvmYQhnGaTLfur9KT+LnVoWLcWA0kPghUUtCA6aD0s3CH0iQIF+19RrKI4BlqbBim2XBktJ6EH77zev/fI1/aLQkhCaruJWYcqGd5Mx0KTTJl4u8YloHn0xb7Jw5XEOWiVqYLAs1r8ooUE1jugBK1fzGczNZF/6gZIyCKQ6E+ETBN91UWx7Sazvs5zoJ/zISmurKm7oU9+kQjrGKKwKzODLCVVmIK43uxyhFBi2IqE5vpQVPInAnKnSuhDQaNRAa31fNMiRfx3IH6AusXxPIuGZQs8BVymgMU0gg0iYj9O5VBdkZPkvjRyiRV7d/4r7iv7dC8Q3mfmweDYmeVJtJIXklgDvAz7tTxAqXeakP75D1BwkwA+sBf2ammXhNY6Yz2WioJJqMkpcOuCFN8MLms6ZHehKoJdM8E8jXwuxmTBMXSZKMwhoRdLb/5P0eq66aHGkm7fMKYIAoC67JVUkO55aPLv5T33q3hjTyiukt6QbJHcbbf48t7UA1Bxz4bmaFJxnf1mMX4RX8m5i+Tb59czcp5LKpCq4Q2Fui2AOQqFRIq7yAxw2AUWl4gI0xX8tGQE7AG3/E7QWfAD0gaXyCtDStJndJ2Lly5VTTZ9Ip/vhMH1T0sPh66gFJBkZ/Sjg6DKAmExRcwDPHe6dxj+ZTRO7MCpQmsDnH0Ado2NXho9+7SdI9xd1DCLnZUxiUV5XpplU5UDOqHXGV4nmjCn3KLXZBEJArP+2UUHTNpF4Bwx3WyYF0S9nG99yE3Svtw5Jsb/sIRaRVlf2C/++ONg3W0RZzwDsTA6tvAOXjEJk/KSkteQ59UW5bHQ/bJEvBDbcIGPRuj6sLCA28Ps/c+aSFHHETvko7oAYwB3UvyLquFBy9eJ2x8oxJ6FS990FCq5QhQ4Uz4y7u2qTOl6sZwgMT9BplXtZ0zHiCFWMVDbVhG0lIXbmzHsoIx+cVAH+P2wQr+FU6pt5EMQHk8wV7lRulGhWSG7Qk6nub2Ng/2/BCs1iWUBpkaqmnJgfG9ZQLz5isi8a/ipheyDbPJBpfFf34WW111ULpP81Kc4t6/r42u9iRZt/dTWLr2tH6niupIFy+mxO1bAF76FeT7nDynHYBq+fZrYKz0fcK7lQCiVN4H+zpb+LK0LalQ7x9IllgcMbpHt+3aX0d+hXZw0ovVoyYXp5BVngCB1uc0DSFsG9kxeHFHUzdsHMVx617Y1j9Yq00nJfCIWHXdKAsOzyANpJJ0+14Q/AGm9ZYi9HhSLvNgY7DKryWyIHp5aMeU2Agw0Sl05/hqDnu1GcK6wmgIekcLMkASkSYIYk93UZP7/YeruQpsKJ+fcnfTzav4hgRv1SoZ/cQgMjhOGoH2pptYQAvEsagQV7grcA/itTI1BQsZ4wk3uhfO71k+bqVzl/Qxyu7qYja7vqnzi4Wi0+eu4KmrnY+RbhLOh0HX/AqZdddo9h05FCTwYZvlsD6Sw64ct+O2I2mzYEp3XVLZ8T3DfYAN/LFHphG76Pe2VGcW4kmJ4pesB3YRUxOc55qZroHetIt2LWOznO8Lvxpzl0XDJ3OhtWixC4Cggt+zScm7W8OyK+/QoivyFMJap0eOjoPfdeilAQOdJeH89PC8fJgw32M90KOMON4x8TULvWKJtcgBIXdevoZ9mbBX2vF/xPyPcLRq4kIbPH96Idfsu9v3v/49vPN9xEWO2zluuJbXqVv7G3EPpnv3uwak727boG+52XATfsWrwVVkLcf3/3w4cvN5U9vf/7w/ubTZ19nwybVxWqkY3yp06/SgzVI2TvFXea/h2fjsPK2ghQgeOAFCgNvOpmob/uHgA9j9h6IJF10kBBCCv8yLfYcOihbaCBQ8XrwPiLdnWg24opc5UAlkmltm6/cXZ9ag2iiGznOcNV8cnRG/KEmcos82wzdZ3eDeYKTHC58ZvPKjp9kypIV6T7GiIJ0ODl1wtB5m1uSdx/fBSWwIS+S4AAyx1E699U5uoXbuwuwJ9KFTkxq2Ff3lgLs5Q2leJ4xQyj+ODyh5jjTkL/1POqjuhIy2Ai9YSZfjVzGMkGDEeQNVktnq9vRp0Ul+ttfvyycat1csVmwXFrYa+HeOuCG0zrgfGA7wxf/B1BLAwQUAAAACAAAACEopNVy7FIAAABVAAAAFwAAAHNjcmlwdHMvZmRpL19faW5pdF9fLnB5Dcs7CoAwDADQ3VOE7O0pRHBx8QQ1TSHYJtIP6O3t9paHiKeNSuyuQDdH2NYdWh/x83AYJNGgJCG7YpEziCaurMQgDaQ8Vvs8VoFfpjHtEXH5AVBLAwQUAAAACAAAACEofFioaXUAAACfAAAAHQAAAHJlc2VhcmNoL2ZkaS9yZXF1aXJlbWVudHMudHh0JY5JEoUgDAX3HCbFpDsOg8LCKoYIob7c/gdddtIveWVknM5p0LAL9CX4/pER/bzWSoHaFpGnnmuIiQUJysImsidMldJ1OGdASdYqxoLzSe+AlSe1sA5yQuD0rdUfo2SWosV7xE7vQ6PBsnDWUaitQhZ2jvwBUEsBAhQAFAAAAAgAAAAhKDfldyhAFAAAqTMAABkAAAAAAAAAAAAAAIABAAAAAHNjcmlwdHMvZmRpX3JvYnVzdG5lc3MucHlQSwECFAAUAAAACAAAACEoGBa1c48RAABXLQAAFQAAAAAAAAAAAAAAgAF3FAAAc2NyaXB0cy9mZGkvbW9kZWxzLnB5UEsBAhQAFAAAAAgAAAAhKBPfHmL8BwAAjRcAABYAAAAAAAAAAAAAAIABOSYAAHNjcmlwdHMvZmRpL3N0b3JhZ2UucHlQSwECFAAUAAAACAAAACEopNVy7FIAAABVAAAAFwAAAAAAAAAAAAAAgAFpLgAAc2NyaXB0cy9mZGkvX19pbml0X18ucHlQSwECFAAUAAAACAAAACEofFioaXUAAACfAAAAHQAAAAAAAAAAAAAAgAHwLgAAcmVzZWFyY2gvZmRpL3JlcXVpcmVtZW50cy50eHRQSwUGAAAAAAUABQBeAQAAoC8AAAAA')
assert hashlib.sha256(raw).hexdigest()=='63930199fe72810a5ce352f7cf7489e4ddff65195548a2782cc6aa4790b8f2f1'
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert z.testzip() is None
    for name in z.namelist():
        assert (PROJECT/name).resolve().is_relative_to(PROJECT.resolve())
    z.extractall(PROJECT)
print('Source SHA256 verified:','63930199fe72810a5ce352f7cf7489e4ddff65195548a2782cc6aa4790b8f2f1')

## Environment (CPU only)
Install the tested pinned baseline dependencies. This notebook uses Python only. Installation does not run any estimator.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--quiet','-r',str(PROJECT/'research/fdi/requirements.txt')],check=True)
import importlib.metadata
VERSIONS={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','statsmodels']}
print(VERSIONS)

## Resume inspection and input authentication
Use the existing study folder. Mounting Drive uses the previously authorized project scope.
Find the exact saved quarterly panel and native national output by their known hashes.
Stop if absent or ambiguous; restore a verified archive rather than reacquiring data or re-estimating.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
STUDY=Path('/content/drive/MyDrive/Brazil FDI Observatory')
EXPECTED={'prepared/quarterly.csv':'10426ad1fef8fcec3a53ff94c1753d289de4faf908437b1ffc787ac3b0d56b8a','models/national.json':'eee06465aa0c4ee366e2476d1083902a4f3a3d1a7713d843e2ad239666a0cc82'}
candidates=[]
for panel in STUDY.rglob('quarterly.csv'):
    run=panel.parent.parent
    if all((run/name).is_file() and hashlib.sha256((run/name).read_bytes()).hexdigest()==v for name,v in EXPECTED.items()):candidates.append(run)
assert candidates,'Matching baseline absent: restore a verified archive. Never rerun it here.'
# Identical verified copies are interchangeable; choose the shortest canonical path and disclose it.
RUN=sorted(candidates,key=lambda p:(len(str(p)),str(p)))[0]
OUTPUT=STUDY/'robustness-extension'
sys.path.insert(0,str(PROJECT/'scripts'))
from fdi_robustness import plan,backup
ROOT=plan(RUN,OUTPUT)
(ROOT/'environment.json').write_text(json.dumps(VERSIONS,indent=2))
print('Verified baseline:',RUN,'Matching copies:',len(candidates),'New identity:',ROOT.name)
for p in sorted(ROOT.glob('*-status.json')):print(p.name,p.read_text())
print('Prepared only; baseline untouched.')

## Execution switches
Keep False for preparation/review. Change to True only when deliberately running this new extension.
Each completed stage is content-hash verified before reuse. Changed code/configuration/input creates a new identity.
The simulation stage is intentionally separate and may take longer; do not use Run All blindly.

In [ ]:
RUN_NEW_CHECKS=False
RUN_SIMULATIONS=False

## Stage: residuals
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_NEW_CHECKS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','residuals'],check=True)
else:
    print('Prepared, not executed:','residuals')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Stage: lp
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_NEW_CHECKS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','lp'],check=True)
else:
    print('Prepared, not executed:','lp')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Stage: saved-draws
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_NEW_CHECKS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','saved-draws'],check=True)
else:
    print('Prepared, not executed:','saved-draws')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Stage: simulation
Outputs are saved in the separate identity. Backups are ZIP-CRC and per-member SHA256 read-back verified.
After this stage, download its backup and independently verify the local copy before proceeding.
No result from this cell is present until the switch is deliberately enabled and execution completes.

In [ ]:
if RUN_SIMULATIONS:
    subprocess.run([sys.executable,str(PROJECT/'scripts/fdi_robustness.py'),'--run',str(RUN),'--output',str(OUTPUT),'--stage','simulation'],check=True)
else:
    print('Prepared, not executed:','simulation')

In [ ]:
# Separate download action: never combine cell execution with a blocking browser download wait.
from google.colab import files
if ROOT.exists() and (ROOT/'robustness-backup.zip').exists():
    CHECK=json.loads((ROOT/'backup-verification.json').read_text())
    assert hashlib.sha256((ROOT/'robustness-backup.zip').read_bytes()).hexdigest()==CHECK['sha256']
    print(CHECK)
    # Deliberately uncomment only after a completed stage:
    # files.download(str(ROOT/'robustness-backup.zip'))

## Release gate
No automatic publication exists. A completed output is not automatically scientifically validated.
Inspect all failures, estimates, interval definitions, assumptions, source hashes and simulated coverage uncertainty.
A new national check cannot cure missing municipal exposure histories. Retain null/inconclusive results.
Upload only validated aggregate exports to the website; keep books, private microdata and research archives private.